<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/FinalWork/EDA_Profile_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
data_path_profiles = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/linkedin_experience_annotated.csv"
df_profiles = pd.read_csv(data_path_profiles)
df_profiles.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,NaN,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,NaN,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,NaN,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0


In [3]:
#Fill active Jobs with current date and unknown where we assume that its the active job when job count = 1 (see later code  in 293)
from datetime import date


df_profiles["endDate"] = df_profiles["endDate"].astype(str)

df_profiles.loc[
    ((df_profiles["status"] == "ACTIVE") | (df_profiles["status"] == "UNKNOWN")) & (df_profiles["endDate"].isin(["nan", "NaT"])),
    "endDate"
] = date.today().strftime("%Y-%m")

In [4]:
#Remove linkedIN  (URL) column

df_profiles.drop(columns=['linkedin'], inplace=True, errors='ignore')
df_profiles.head()

,organization,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,Prokurist,2019-08,2026-01,ACTIVE,Other,Management,0
1,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0
2,Depot4Design GmbH,Betriebswirtin,2019-07,2026-01,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,Prokuristin,2019-07,2026-01,ACTIVE,Other,Management,0
4,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0


In [17]:
#Regularize startDate for job_duration_years
import pandas as pd
import re

def normalize_startdate(val):
    if pd.isna(val):
        return pd.NA

    val = str(val).strip()

    # Case 1: YYYY-MM (bereits korrekt)
    if re.fullmatch(r"\d{4}-\d{2}", val):
        return val

    # Case 2: YYYY \u2192 erg\u00e4nze Januar
    if re.fullmatch(r"\d{4}", val):
        return f"{val}-01"

    # alles andere \u2192 missing
    return pd.NA


df_profiles["startDate"] = df_profiles["startDate"].apply(normalize_startdate)

import pandas as pd
import re

def normalize_enddate(val):
    if pd.isna(val):
        return pd.NA

    val = str(val).strip()

    # Case 1: YYYY-MM (bereits korrekt)
    if re.fullmatch(r"\d{4}-\d{2}", val):
        return val

    # Case 2: YYYY → ergänze Januar
    if re.fullmatch(r"\d{4}", val):
        return f"{val}-01"

    # alles andere → missing
    return pd.NA


df_profiles["endDate"] = df_profiles["endDate"].apply(normalize_enddate)


In [16]:
# Imputing strategy for Unknown and missing startDate where there is only one (current job)

df_profiles["startDate"] = df_profiles["startDate"].replace(
    ["", "unknown", "novalue", None],
    pd.NA
)

job_counts = df_profiles.groupby("person_id").size()
df_profiles["job_count"] = df_profiles["person_id"].map(job_counts)

# Calculate job_duration_years here before using it
# Use errors='coerce' to handle non-conforming date strings gracefully
start_temp = pd.to_datetime(df_profiles["startDate"], format="%Y-%m", errors='coerce')
end_temp   = pd.to_datetime(df_profiles["endDate"],   format="%Y-%m", errors='coerce')
df_profiles["job_duration_years"] = (end_temp - start_temp).dt.days / 365

#Median Job Duration for imputing
median_duration_years = df_profiles["job_duration_years"].median()
median_duration_months = int(round(median_duration_years * 12))

mask = (
    df_profiles["startDate"].isna() & # Corrected 'df' to 'df_profiles'
    (df_profiles["job_count"] == 1)
)

# endDate als Referenz, sonst heutiges Datum
reference_date = pd.to_datetime(
    df_profiles.loc[mask, "endDate"], # Corrected 'df' to 'df_profiles'
    format="%Y-%m",
    errors="coerce"
).fillna(pd.Timestamp.today())

imputed_start = reference_date - pd.DateOffset(months=median_duration_months)

df_profiles.loc[mask, "startDate"] = imputed_start.dt.strftime("%Y-%m")

In [13]:
#Drop Rest where theres no start Date and more than one job count (only 23)

df_profiles = df_profiles.drop(
    df_profiles[
        df_profiles["startDate"].isna() &
        (df_profiles["job_count"] > 1)
    ].index
)
#and change status from unknown for imputed to active
df_profiles.loc[
    (df_profiles["status"].str.lower() == "unknown"),
    "status"
] = "ACTIVE"

In [15]:
#Did we successfully change all Unknown to Active if theres only one job? - Yes, no unknown left
df_profiles["status"].value_counts(dropna=False)

,count
status,
INACTIVE,1897
ACTIVE,718


In [18]:
#Check for missing values
df_profiles.isna().sum()

,0
organization,0
position,0
startDate,0
endDate,0
status,0
department,0
seniority,0
person_id,0
job_count,0
job_duration_years,0


In [10]:
#Ready to encode dataframe:

df_profiles.head(75)

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2026-01,ACTIVE,Other,Management,0,6,6.424658
1,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0,6,6.509589
2,Depot4Design GmbH,Betriebswirtin,2019-07,2026-01,ACTIVE,Other,Professional,0,6,6.509589
3,Depot4Design GmbH,Prokuristin,2019-07,2026-01,ACTIVE,Other,Management,0,6,6.509589
4,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0,6,6.509589
...,...,...,...,...,...,...,...,...,...,...
70,Lichtenberg School,"Teacher: History, Ethics, French",2008-08,2017-07,INACTIVE,Other,Professional,20,6,8.920548
71,German School Beijing (China),"Teacher: History, Ethics, French",2002-08,2008-07,INACTIVE,Other,Professional,20,6,5.920548
72,"Universities: Chuncheon, Hannam, Hongik (South...",Dr. phil. - German Studies,1990-02,1999-01,INACTIVE,Other,Professional,20,6,8.920548
73,Thurm GmbH,"Geschäftsführer, CMO",2014-07,2026-01,ACTIVE,Marketing,Management,21,3,11.512329


In [11]:
#Download Code for Team Members
#df_profiles.to_csv("df_profiles.csv", index=False)

#from google.colab import files
#files.download("df_profiles.csv")

